In [2]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [3]:
from toxic_comments.config import PROCESSED_DATA_DIR
df = pd.read_csv(PROCESSED_DATA_DIR / "train_clean.csv")

In [4]:
df[df["is_empty_heavy"] != 0]

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,comment_light,comment_heavy,...,punct_ratio,n_newlines,n_urls,n_ips,n_you,n_masked_words,has_shouting,is_empty_light,is_empty_heavy,is_non_latin
2405,067638a445ccd93b,"Here, here and here.",0,0,0,0,0,0,"Here, here and here.",NaN,...,0.100000,0,0,0,0,0,0,0,1,1
3987,0aa6f3529219b37e,From here\n\nFrom here 160.80.2.8,0,0,0,0,0,0,From here From here,NaN,...,0.096774,2,0,1,0,0,0,0,1,1
4479,0bed2196c873636d,1993\n\n1994\n\n1995\n\n1996\n\n1997\n\n1998\n...,0,0,0,0,0,0,1993 1994 1995 1996 1997 1998 1999 2000 2001 2...,NaN,...,0.000000,20,0,0,0,0,0,0,1,1
6296,10d0c3263b52a057,193.61.111.53 15:00,0,0,0,0,0,0,15:00,NaN,...,0.200000,0,0,1,0,0,0,0,1,1
8841,1776b3bcff81788b,What is I 78.146.102.144,0,0,0,0,0,0,What is I,NaN,...,0.125000,0,0,1,0,0,0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148773,5352b339650c4138,she did 76.122.79.82,0,0,0,0,0,0,she did,NaN,...,0.150000,0,0,1,0,0,0,0,1,1
148947,5631d491869e9f64,"- 00:49, 14 Feb 2005 (UTC)",0,0,0,0,0,0,-,NaN,...,0.185185,0,0,0,0,0,0,0,1,1
151286,7c1a57063b5c14d5,10 - 2010 04 08 to 2010 05 12,0,0,0,0,0,0,10 - 2010 04 08 to 2010 05 12,NaN,...,0.034483,0,0,0,0,0,0,0,1,1
153221,9af7112c4d554edb,which is OVER 9000 OVER 9000 OVER 9000 OVER 90...,0,0,0,0,0,0,which is OVER 9000 OVER 9000 OVER 9000 OVER 90...,NaN,...,0.000000,0,0,0,0,0,1,0,1,1


### create k-folds

In [13]:
from toxic_comments.splits import save_kfold_datasets

files = save_kfold_datasets(df)
files

[PosixPath('/home/jkl0909/minhhuyen/toxic-comments/data/k_fold/fold_1/train.csv'),
 PosixPath('/home/jkl0909/minhhuyen/toxic-comments/data/k_fold/fold_1/test.csv'),
 PosixPath('/home/jkl0909/minhhuyen/toxic-comments/data/k_fold/fold_2/train.csv'),
 PosixPath('/home/jkl0909/minhhuyen/toxic-comments/data/k_fold/fold_2/test.csv'),
 PosixPath('/home/jkl0909/minhhuyen/toxic-comments/data/k_fold/fold_3/train.csv'),
 PosixPath('/home/jkl0909/minhhuyen/toxic-comments/data/k_fold/fold_3/test.csv'),
 PosixPath('/home/jkl0909/minhhuyen/toxic-comments/data/k_fold/fold_4/train.csv'),
 PosixPath('/home/jkl0909/minhhuyen/toxic-comments/data/k_fold/fold_4/test.csv'),
 PosixPath('/home/jkl0909/minhhuyen/toxic-comments/data/k_fold/fold_5/train.csv'),
 PosixPath('/home/jkl0909/minhhuyen/toxic-comments/data/k_fold/fold_5/test.csv')]

In [14]:
from toxic_comments.config import K_FOLD_DATA_DIR, LABEL_COLUMNS

rows = []
for fold_index in range(1, 6):
    fold_dir = K_FOLD_DATA_DIR / f"fold_{fold_index}"
    for split_name in ["train", "test"]:
        split_path = fold_dir / f"{split_name}.csv"
        split_df = pd.read_csv(split_path)

        row = {
            "fold": fold_index,
            "split": split_name,
            "rows": len(split_df),
        }
        row.update({label: int(split_df[label].sum()) for label in LABEL_COLUMNS})
        rows.append(row)

fold_label_summary = pd.DataFrame(rows)
fold_label_summary


,fold,split,rows,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,1,train,127580,12230,1273,6756,381,6300,1125
1,1,test,31896,3060,321,1690,97,1574,280
2,2,train,127581,12231,1275,6756,383,6299,1123
3,2,test,31895,3059,319,1690,95,1575,282
4,3,train,127581,12233,1275,6755,384,6299,1125
5,3,test,31895,3057,319,1691,94,1575,280
6,4,train,127581,12233,1276,6759,382,6297,1125
7,4,test,31895,3057,318,1687,96,1577,280
8,5,train,127581,12233,1277,6758,382,6301,1122
9,5,test,31895,3057,317,1688,96,1573,283


### divide into train-validation-test

In [9]:
from toxic_comments.splits import train_validation_test_split
from toxic_comments.config import LABEL_COLUMNS

train_df, validation_df, test_df = train_validation_test_split(
    df,
    validation_size=0.1,
    test_size=0.1,
    random_state=42,
    stratified=True,
)

rows = []
for split_name, split_df in [
    ("train", train_df),
    ("validation", validation_df),
    ("test", test_df),
]:
    row = {
        "split": split_name,
        "rows": len(split_df),
    }
    row.update({label: int(split_df[label].sum()) for label in LABEL_COLUMNS})
    rows.append(row)

holdout_label_summary = pd.DataFrame(rows)
holdout_label_summary


,split,rows,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,train,127580,12232,1275,6757,384,6299,1123
1,validation,15948,1529,162,845,44,786,143
2,test,15948,1529,157,844,50,789,139


### Test training with tf-idf

In [10]:
from toxic_comments.config import MODELS_DIR
from toxic_comments.train import save_model, train_model

model_name = "tfidf_logistic_regression"
text_column = "comment_heavy"

estimator = train_model(
    data=train_df,
    model_name=model_name,
    text_column=text_column,
    max_features=20_000,
)

model_path = save_model(
    estimator,
    MODELS_DIR / "notebook_test" / f"{model_name}.joblib",
)

print(f"Saved model: {model_path}")


Saved model: /home/jkl0909/minhhuyen/toxic-comments/models/notebook_test/tfidf_logistic_regression.joblib


In [11]:
from toxic_comments.config import RESULTS_DIR
from toxic_comments.train import evaluate_model

validation_metrics = {
    "model_name": model_name,
    "split": "validation",
    "rows": len(validation_df),
}
validation_metrics.update(
    evaluate_model(estimator, validation_df, text_column=text_column)
)

validation_metrics_df = pd.DataFrame([validation_metrics])
metrics_path = RESULTS_DIR / "notebook_test" / f"{model_name}_validation_metrics.csv"
metrics_path.parent.mkdir(parents=True, exist_ok=True)
validation_metrics_df.to_csv(metrics_path, index=False)

print(f"Saved validation metrics: {metrics_path}")
validation_metrics_df


Saved validation metrics: /home/jkl0909/minhhuyen/toxic-comments/results/notebook_test/tfidf_logistic_regression_validation_metrics.csv


,model_name,split,rows,subset_accuracy,hamming_loss,micro_precision,micro_recall,micro_f1,macro_f1,micro_roc_auc
0,tfidf_logistic_regression,validation,15948,0.878731,0.029126,0.568449,0.854374,0.682682,0.559379,0.982115


### to download word2vec embeddings

In [3]:
import gensim.downloader as api
from toxic_comments.config import EMBEDDINGS_DIR

path = EMBEDDINGS_DIR / "word2vec" / "GoogleNews-vectors-negative300.txt"
path.parent.mkdir(parents=True, exist_ok=True)

vectors = api.load("word2vec-google-news-300")
vectors.save_word2vec_format(str(path), binary=False)

[==================================================] 100.0% 1662.8/1662.8MB downloaded


## Mean pooling: five embeddings

Reuse train_df and validation_df from the holdout split above.
Each experiment uses pretrained embeddings, pooling, StandardScaler and LogisticRegression.
The test split is reserved for the final selected model. The download cell above does not need to run again.
Models include their embedding data, so saved pipelines can occupy several GB.


In [13]:
import gc
import joblib
from time import perf_counter

from toxic_comments.config import LIGHT_TEXT_COLUMN, MODELS_DIR, RESULTS_DIR
from toxic_comments.train import train_model, save_model, evaluate_model

embedding_model_names = [
    "fasttext_logistic_regression",
    "glove_twitter_logistic_regression",
    "bpemb_logistic_regression",
    "word2vec_logistic_regression",
    "lexvec_logistic_regression",
]
embedding_text_column = "comment_heavy"
embedding_pooling = "max_attention" #"mean", "max", "attention", "max_attention"
max_embedding_vectors = None  # Full word vocabularies; BPEmb always uses all subwords.
embedding_model_dir = MODELS_DIR / "notebook_test" / "embeddings_max_attention"
embedding_results_dir = RESULTS_DIR / "notebook_test" / "embeddings_max_attention"

for split_frame in (train_df, validation_df):
    if embedding_text_column not in split_frame:
        raise ValueError(f"Missing column: {embedding_text_column}")


In [14]:
embedding_training_rows = []
embedding_model_paths = {}

for embedding_model_name in embedding_model_names:
    print(f"Training {embedding_model_name}", flush=True)
    started = perf_counter()
    embedding_estimator = train_model(
        data=train_df,
        model_name=embedding_model_name,
        text_column=embedding_text_column,
        embedding_pooling=embedding_pooling,
        max_embedding_vectors=max_embedding_vectors,
    )
    fit_seconds = perf_counter() - started
    embedding_model_path = save_model(
        embedding_estimator,
        embedding_model_dir / f"{embedding_model_name}.joblib",
    )
    embedding_model_paths[embedding_model_name] = embedding_model_path
    embedding_training_rows.append({
        "model_name": embedding_model_name,
        "pooling": embedding_pooling,
        "text_column": embedding_text_column,
        "train_rows": len(train_df),
        "fit_seconds": fit_seconds,
        "model_path": str(embedding_model_path),
        "max_embedding_vectors": max_embedding_vectors,
    })
    del embedding_estimator
    gc.collect()
    print(f"Saved: {embedding_model_path}", flush=True)

embedding_training_summary = pd.DataFrame(embedding_training_rows)
embedding_results_dir.mkdir(parents=True, exist_ok=True)
embedding_training_summary.to_csv(
    embedding_results_dir / "training_summary.csv", index=False,
)
display(embedding_training_summary)


Training fasttext_logistic_regression
Saved: /home/jkl0909/minhhuyen/toxic-comments/models/notebook_test/embeddings_max_attention/fasttext_logistic_regression.joblib
Training glove_twitter_logistic_regression
Saved: /home/jkl0909/minhhuyen/toxic-comments/models/notebook_test/embeddings_max_attention/glove_twitter_logistic_regression.joblib
Training bpemb_logistic_regression
Saved: /home/jkl0909/minhhuyen/toxic-comments/models/notebook_test/embeddings_max_attention/bpemb_logistic_regression.joblib
Training word2vec_logistic_regression
Saved: /home/jkl0909/minhhuyen/toxic-comments/models/notebook_test/embeddings_max_attention/word2vec_logistic_regression.joblib
Training lexvec_logistic_regression
Saved: /home/jkl0909/minhhuyen/toxic-comments/models/notebook_test/embeddings_max_attention/lexvec_logistic_regression.joblib


,model_name,pooling,text_column,train_rows,fit_seconds,model_path,max_embedding_vectors
0,fasttext_logistic_regression,max_attention,comment_heavy,127580,268.912648,/home/jkl0909/minhhuyen/toxic-comments/models/...,None
1,glove_twitter_logistic_regression,max_attention,comment_heavy,127580,287.463073,/home/jkl0909/minhhuyen/toxic-comments/models/...,None
2,bpemb_logistic_regression,max_attention,comment_heavy,127580,633.311837,/home/jkl0909/minhhuyen/toxic-comments/models/...,None
3,word2vec_logistic_regression,max_attention,comment_heavy,127580,333.806045,/home/jkl0909/minhhuyen/toxic-comments/models/...,None
4,lexvec_logistic_regression,max_attention,comment_heavy,127580,264.240032,/home/jkl0909/minhhuyen/toxic-comments/models/...,None


In [15]:
embedding_validation_rows = []

for embedding_model_name, embedding_model_path in embedding_model_paths.items():
    print(f"Validating {embedding_model_name}", flush=True)
    embedding_estimator = joblib.load(embedding_model_path)
    row = {
        "model_name": embedding_model_name,
        "pooling": embedding_pooling,
        "text_column": embedding_text_column,
        "split": "validation",
        "rows": len(validation_df),
    }
    row.update(evaluate_model(
        embedding_estimator,
        validation_df,
        text_column=embedding_text_column,
    ))
    embedding_validation_rows.append(row)
    del embedding_estimator
    gc.collect()

embedding_validation_summary = (
    pd.DataFrame(embedding_validation_rows)
    .sort_values("micro_roc_auc", ascending=False, na_position="last")
    .reset_index(drop=True)
)
embedding_validation_summary.to_csv(
    embedding_results_dir / "validation_metrics.csv", index=False,
)
display(embedding_validation_summary)


Validating fasttext_logistic_regression
Validating glove_twitter_logistic_regression
Validating bpemb_logistic_regression
Validating word2vec_logistic_regression
Validating lexvec_logistic_regression


,model_name,pooling,text_column,split,rows,subset_accuracy,hamming_loss,micro_precision,micro_recall,micro_f1,macro_f1,micro_roc_auc
0,fasttext_logistic_regression,max_attention,comment_heavy,validation,15948,0.832205,0.048512,0.424517,0.907951,0.578536,0.452480,0.979675
1,word2vec_logistic_regression,max_attention,comment_heavy,validation,15948,0.816403,0.056548,0.383939,0.896552,0.537640,0.422116,0.974122
2,lexvec_logistic_regression,max_attention,comment_heavy,validation,15948,0.810635,0.058325,0.375750,0.892847,0.528910,0.407785,0.971788
3,glove_twitter_logistic_regression,max_attention,comment_heavy,validation,15948,0.794143,0.069141,0.332290,0.877173,0.481992,0.378837,0.966080
4,bpemb_logistic_regression,max_attention,comment_heavy,validation,15948,0.739403,0.094777,0.259432,0.854374,0.398009,0.311903,0.949454
